# TMC-PINN Experiments

Run cells top to bottom. Each PDE section is independent — run whichever you need.

**Epoch budget (matches Xu et al. exactly):**
- reaction: 2,000  (switch @ 1,000)
- wave: 10,000  (switch @ 5,000)
- ac: 10,000  (switch @ 5,000)
- convection: 50,000  (switch @ 25,000)

**Architecture:** 512x4, Xavier init, Tanh (790,017 params)

In [ ]:
# Cell 1 — GPU check
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU')

In [ ]:
# Cell 2 — Clone / update repo
import os
if not os.path.exists('PInnns'):
    os.system('git clone https://github.com/michae6345-crypto/PInnns.git')
else:
    os.system('cd PInnns && git pull origin main')
os.chdir('PInnns')
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Cell 3 — Load train_pinn
import importlib.util, sys
if 'train_pinn' in sys.modules:
    del sys.modules['train_pinn']
spec = importlib.util.spec_from_file_location('train_pinn', './train_pinn.py')
tp   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tp)
print('Loaded. Epoch budget:', tp.PDE_EPOCHS)

In [ ]:
# Cell 4 — Smoke test (50 epochs, ~1 min)
# Run this first to confirm everything works.
tp.main(pde='reaction', dtype_start='fp64', optim_start='lbfgs', total_epochs=50)
import os
print('\nSmoke test PASSED. Files:', os.listdir('./results/reaction/fp64lbfgs/'))

---
## Run PDEs
Each cell runs all 7 conditions for one PDE. Run whichever you need.
`skip_existing=True` means already-completed conditions are skipped automatically.

In [ ]:
# Cell 5 — REACTION  (~15-20 min, 2,000 epochs x 7 conditions)
tp.run_all(
    pdes          = ['reaction'],
    out_dir       = './results',
    device        = 'cuda:0',
    skip_existing = True,
)

In [ ]:
# Cell 6 — WAVE  (~2-4 hrs, 10,000 epochs x 7 conditions)
# u_tt = 4*u_xx, IC = sin(pi*x) + 0.5*sin(3*pi*x), 101x101 grid
tp.run_all(
    pdes          = ['wave'],
    out_dir       = './results',
    device        = 'cuda:0',
    skip_existing = True,
)

In [ ]:
# Cell 7 — ALLEN-CAHN  (~2-4 hrs, 10,000 epochs x 7 conditions)
# IC is analytic x^2*cos(pi*x). allen_cahn.mat only needed for L2 evaluation.
import os
print('allen_cahn.mat present:', os.path.exists('allen_cahn.mat'))
tp.run_all(
    pdes          = ['ac'],
    out_dir       = './results',
    device        = 'cuda:0',
    mat_path      = './allen_cahn.mat',
    skip_existing = True,
)

In [ ]:
# Cell 8 — CONVECTION  (~8-15 hrs, 50,000 epochs x 7 conditions)
# beta=50, 401x401 training grid. Run last — by far the longest.
tp.run_all(
    pdes          = ['convection'],
    out_dir       = './results',
    device        = 'cuda:0',
    skip_existing = True,
)

---
## Utilities

In [ ]:
# Cell 9 — Check results summary
import os, pandas as pd

conditions = [
    'fp64lbfgs','fp64adam','fp32lbfgs','fp32adam',
    'fp32adam_to_fp64adam','fp32lbfgs_to_fp64lbfgs','fp32adam_to_fp64lbfgs'
]
pdes = ['reaction','wave','ac','convection']

print(f'{"PDE":<12} {"Condition":<35} {"L2 error":<14} {"Time(s)"}')
print('-'*75)
for pde in pdes:
    for cond in conditions:
        path = f'./results/{pde}/{cond}/{pde}_PINN_{cond}_eval.csv'
        if os.path.exists(path):
            try:
                row = pd.read_csv(path).iloc[-1]
                print(f'{pde:<12} {cond:<35} {row["L2_rel"]:<14.4e} {row["total_time_s"]:.0f}')
            except Exception:
                print(f'{pde:<12} {cond:<35} ERROR')
        else:
            print(f'{pde:<12} {cond:<35} NOT RUN')

In [ ]:
# Cell 10 — Generate paper figures
import importlib.util, os
spec = importlib.util.spec_from_file_location('ag', './aggregate_results.py')
ag   = importlib.util.module_from_spec(spec); spec.loader.exec_module(ag)
os.makedirs('./paper_figures', exist_ok=True)
eval_df = ag.load_eval_logs('./results')
loss_df = ag.load_loss_logs('./results')
ag.build_summary_table(eval_df, './paper_figures')
ag.plot_loss_curves(loss_df, eval_df, './paper_figures')
ag.plot_l2_bars(eval_df, './paper_figures')
ag.plot_timing_bars(eval_df, './paper_figures')
print('Figures:', os.listdir('./paper_figures/'))

In [ ]:
# Cell 11 — Backup to zip (download before closing Lambda)
import zipfile, datetime, os
ts  = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zfn = f'results_backup_{ts}.zip'
with zipfile.ZipFile(zfn, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['./results', './paper_figures']:
        for root, dirs, files in os.walk(folder):
            for file in files:
                zf.write(os.path.join(root, file))
print(f'Backup: {zfn}  ({os.path.getsize(zfn)/1e6:.1f} MB)')
print('Download from Jupyter file browser then upload to Drive.')